# Notebook for å autentisere seg mot skyporten og hente ut data fra share

In [1]:
! pip install -r requirements.txt

In [2]:
import os
import json
import subprocess
import delta_sharing
from google.cloud import storage

from src.auth import generate_access_token
from src.utils import (
    fetch_config_share,
    create_credentials_config,
)

In [3]:

project_id = "innsikt-data-dev-ec28"
project_num = "614733074632"
provider_full_identifier = f"projects/{project_num}/locations/global/workloadIdentityPools/skyporten-bi-dev/providers/skyporten-bi-provider-dev"
random_id = "9ivj"
schema_name = "matrikkel_silver_v1_ext"
share_name = f"{random_id}-dev"

CREDENTIALS_PATH = "credentials.json"
TOKEN_PATH = "tmp_maskinporten_token.txt"
CONFIG_PATH = "configs/config.json"
SOURCE_PATH = "share/config.share" #filen du leser fra dersom du har en gyldig config.share, eller ønsker å skrive til dersom share har utløpt

In [4]:
create_credentials_config(provider_full_identifier, TOKEN_PATH, CREDENTIALS_PATH)

Created credential configuration file [credentials.json].


In [5]:
with open(CONFIG_PATH, 'r') as file:
    config = json.load(file)

token = generate_access_token(
    kid=config.get('kid'),
    scope=config.get('scope'),
    certname=config.get('certname'),
    audience=config.get('audience'),
    client_id=config.get('client_id'),
    token_url=config.get('url'),
)

s = token.get("access_token", "")

print(f"Generated token: {s}")

with open(TOKEN_PATH, 'w') as file:
    file.write(s)



Generated token: eyJraWQiOiJiZFhMRVduRGpMSGpwRThPZnl5TUp4UlJLbVo3MUxCOHUxeUREbVBpdVQwIiwiYWxnIjoiUlMyNTYifQ.eyJhdWQiOiJodHRwczovL3NreXBvcnRlbi5rYXJ0dmVyay5ubyIsInN1YiI6IjAxOTI6OTcxMDQwMjM4O2thcnR2ZXJrOm1hdHJpa2tlbC5iZXJldHRpZ2V0aW50ZXJlc3NlIiwic2NvcGUiOiJrYXJ0dmVyazptYXRyaWtrZWwuYmVyZXR0aWdldGludGVyZXNzZSIsImlzcyI6Imh0dHBzOi8vdGVzdC5za3kubWFza2lucG9ydGVuLm5vIiwiY2xpZW50X2FtciI6InByaXZhdGVfa2V5X2p3dCIsInRva2VuX3R5cGUiOiJCZWFyZXIiLCJleHAiOjE3NTk5MjA1NTMsImlhdCI6MTc1OTkyMDUyMywiY2xpZW50X2lkIjoiMWIxOWY0N2YtODBmNC00ZTM0LWE3ODMtODQ4ZWM0YjI5YTU2IiwianRpIjoiNTNzRmxJU0pWSDQzWlhvcE0tMzdLeXZnbW5FVy1lTFhWYXZEN3VfVXM0WSIsImNvbnN1bWVyIjp7ImF1dGhvcml0eSI6ImlzbzY1MjMtYWN0b3JpZC11cGlzIiwiSUQiOiIwMTkyOjk3MTA0MDIzOCJ9fQ.p3A5Hf_m1QKl-bEEM3bR3tpDEpqZXR9L7c1S7hT9itwcQsKcPo9I15cYKRxmE7BUUKQ4AIBGuKx0zd8j5-2ttJ39xv_T89cNLeDviA_iNUixmNthmu5uhKItZv7aBf5f4pgoBkwpzQo8bqTniUc7Y8AVtbayMkCctEPu8HQwwJPEmaXSC3gIrDExagjuFiRrak9fS2zfVr91MFg_NcAch30IHcyc93s0MCSq-lk52GC4h45qtDhfmjzdZUxKN-vjB2cTMdwUtxw5jjMNr4pUA1yKr_K8P3RPL

In [6]:
os.makedirs("share", exist_ok=True)
bucket_id = f"sp-{project_id}-{random_id}"
fetch_config_share(project_id, bucket_id, SOURCE_PATH, CREDENTIALS_PATH)

config is expired or missing, downloading a new one...
Downloaded storage object sp-innsikt-data-dev-ec28-9ivj from bucket config.share to local file share/config.share.


In [7]:
sharing_client = delta_sharing.SharingClient(SOURCE_PATH)

tables = []
try:
    tables = sharing_client.list_all_tables()
except:
    print("Sharen har ikke tilgang til noen tabeller")
    exit(1)

for table in tables:
    print(table.name)

dim_kulturminneartkode
keys_encrypted_9ivj
dim_kulturminner
dim_kulturminner_encrypted
dim_adresse
dim_adressereferansekode
dim_adressetilleggsnavnkildekode
fact_kulturminner
fact_kulturminner_historical


In [8]:
table_name = tables[0].name #henter første tabell som er tilgjengelig i share, erstatt med ønsket tabellnavn 
table_url = f"{SOURCE_PATH}#{share_name}.{schema_name}.{table_name}"
data = delta_sharing.load_as_pandas(table_url)
print(data.head())

   kulturminneartKodeId kodeverdi         navn_bokmaal navn_nynorsk  \
0                  5501     20261      Militære anlegg         None   
1                  5502     20262             Flyplass         None   
2                  5503     20263  Hellegropslokalitet         None   
3                  5504     20264            Gårdshaug         None   
4                  5505     30001                Ferge         None   

              from_datetime                      to_datetime  \
0 1970-01-01 00:00:00+00:00 1815-03-31 05:56:08.066278+00:00   
1 1970-01-01 00:00:00+00:00 1815-03-31 05:56:08.066278+00:00   
2 1970-01-01 00:00:00+00:00 1815-03-31 05:56:08.066278+00:00   
3 1970-01-01 00:00:00+00:00 1815-03-31 05:56:08.066278+00:00   
4 1970-01-01 00:00:00+00:00 1815-03-31 05:56:08.066278+00:00   

        zx_ingest_timestamp  zx_ingest_file_name  
0 2024-04-09 00:00:00+00:00  09-04-24-objekter-0  
1 2024-04-09 00:00:00+00:00  09-04-24-objekter-0  
2 2024-04-09 00:00:00+00:00  09-04-

In [9]:
table_url = f"{SOURCE_PATH}#{share_name}.{schema_name}.{table_name}"
metadata = delta_sharing.get_table_metadata(table_url)

print(metadata)

Metadata(id='109e75d6-ea1a-4cef-9d74-bb14ff506bcf', name=None, description='None', format=Format(provider='parquet', options={}), schema_string='{"type":"struct","fields":[{"name":"kulturminneartKodeId","type":"long","nullable":true,"metadata":{}},{"name":"kodeverdi","type":"string","nullable":true,"metadata":{}},{"name":"navn_bokmaal","type":"string","nullable":true,"metadata":{}},{"name":"navn_nynorsk","type":"string","nullable":true,"metadata":{}},{"name":"from_datetime","type":"timestamp","nullable":true,"metadata":{}},{"name":"to_datetime","type":"timestamp","nullable":true,"metadata":{}},{"name":"zx_ingest_timestamp","type":"timestamp","nullable":true,"metadata":{}},{"name":"zx_ingest_file_name","type":"string","nullable":true,"metadata":{}}]}', configuration={'delta.enableChangeDataFeed': 'true', 'delta.enableDeletionVectors': 'false'}, partition_columns=[], version=None, size=6454, num_files=1, created_time=1716991207419)
